# Neuro-Symbolic Agentic AI for Autonomous Finance, Accounting, Audit and Compliance: An Autonomous ITC Recovery and Vendor Compliance Agent(POC)

## Architecture invariant

**The LLM never decides eligibility. The rule engine is the sole authority on every verdict.**

Two layers, strictly separated:
- **Layer 1 — LLM (perception & explanation only):** `Mistral 7B`, served locally via `Ollama`. Used exclusively to narrate a verdict *after* the rule engine has already decided it. It receives the verdict and reason chain as fixed input and cannot alter them.
- **Layer 2 — Rule Engine (decision, deterministic):** a YAML-driven catalogue of legal rules under Section 16(2), 16(4), 17(5), and Rule 36(4) of the CGST Act 2017. This is the *only* place in the entire system where an eligibility verdict is produced.

If the LLM hallucinates, mislabels, or gets confused (see the Section 17(5) example later in this notebook, where it once mistyped "ITC" as "International Trade Centre"), it cannot corrupt a verdict — because it was never asked to produce one.

## Files this notebook depends on

**Rule catalogue** (`itc/rules/catalogue/`) — 4 YAML rule definitions, loaded once at the top of this notebook:

- `section_16_2.yaml` — core eligibility, evaluated as four sequential sub-clauses (a→d), short-circuiting on the first failure:
  - **16(2)(a)** — buyer is a registered person under GST
  - **16(2)(b)** — goods/services were actually received (not just invoiced)
  - **16(2)(c)** — supplier filed GSTR-1 reporting the transaction
  - **16(2)(d)** — tax was actually paid, evidenced by presence in GSTR-2B
- `section_16_4.yaml` — time-bar: ITC must be claimed by 30 November following the end of the financial year (Apr–Mar cycle), else `time_barred`
- `section_17_5.yaml` — blocked credit categories (motor vehicles, food & beverage, club memberships, etc.), matched via keywords against `item_description_clean`; permanently `blocked` regardless of any other condition
- `rule_36_4.yaml` — provisional ITC date-gate: pre-2022 invoices missing from GSTR-2B get `provisional`; abolished for tax periods ≥ 01/2022 (Notification 40/2021-CT), which forces an override to `ineligible`

**Domain models** (`itc/domain/`):
- `facts.py` — `InvoiceFacts` (the rule engine's only input) and `MatchResult`
- `verdict.py` — `Verdict`, `VerdictType`, `ReasonStep` (the rule engine's only output)

**Ingestion** (`itc/ingestion/`):
- `gstr2b.py` — parses synthetic GSTR-2B JSON (government portal data) into structured entries
- `purchase_register.py` — parses the tenant's purchase register Excel file via a configurable `ColumnMapping`, since real-world tally exports use inconsistent headers

**Rules engine** (`itc/rules/`):
- `loader.py` — loads and validates the YAML catalogue into `RuleCatalogue`
- `engine.py` — `evaluate(facts, catalogue)`: the single function where a verdict is decided

**Orchestration** (`itc/agents/`):
- `reconciliation.py` — matches purchase register rows against GSTR-2B entries, builds `InvoiceFacts`, calls the rule engine, and packages non-eligible results into `Case`s

**LLM integration** (`itc/intelligence/`):
- `gateway.py` — `AbstractLLMGateway` contract + `StubLLMGateway` (M0 placeholder, unused in this notebook)
- `ollama_gateway.py` — `OllamaLLMGateway`, the real implementation used here: calls local Ollama, validates the response against a Pydantic schema, retries once on failure
- `schemas.py` — `VerdictExplanation`, the schema the LLM's output must validate against
- `models.py` — `LLMTrace`, an audit record written for every LLM call (see the final section of this notebook)

**Synthetic data generators** (`itc/scripts/`), run separately before this notebook:
- `generate_gstr2b.py` — produces `fixtures/gstr2b_{tenant}_{period}.json`
- `generate_purchase_register.py` — produces `fixtures/purchase_register_{tenant}.xlsx`, with ~5% deliberately unmatched invoices and amount drift injected, to exercise every rule branch


## Data flow through this notebook

`generate_gstr2b.py`, `generate_purchase_register.py` (run once, offline, before this notebook)

▼

`fixtures/*.json`, `fixtures/*.xlsx`

▼

`gstr2b.py` + `purchase_register.py` (Ingestion — parse raw files into structured entries)

▼

`reconciliation.py` (Match purchase register rows ↔ GSTR-2B entries by GSTIN + invoice number)

▼

`InvoiceFacts` (`facts.py` — the neutral, rule-engine-ready representation of one invoice)

▼

`engine.py` :: `evaluate(facts, catalogue)` ◄── `rules/catalogue/*.yaml` (Section 16(2), 16(4), 17(5), Rule 36(4))

> ***** THIS is the only step that decides eligibility. Nothing before or after it can override this verdict. *****

▼

`Verdict` + `ReasonStep` chain (`verdict.py`)

▼

`Case` (`reconciliation.py` — packages verdict + facts for every non-eligible invoice)

▼

`ollama_gateway.py` :: `OllamaLLMGateway.call()` ◄── Mistral 7B via Ollama (`localhost:11434`) — explains the verdict in plain English, cannot change it

▼

`VerdictExplanation` (`schemas.py` — validated LLM output)

▼

Printed output below, + `LLMTrace` audit record for every call

## 1. Setup — make the `itc` package importable

In [1]:
import sys
from itc.rules.engine import evaluate
from pathlib import Path

# Notebook lives in backend/ alongside itc/ and scripts/ — adjust if you move it
sys.path.insert(0, str(Path.cwd()))

from datetime import date

from itc.domain.facts import InvoiceFacts
from itc.ingestion.gstr2b import parse_gstr2b
from itc.ingestion.purchase_register import parse_register, mapping_from_tenant_profile
from itc.rules.loader import load_catalogue
from itc.agents.reconciliation import reconcile
from itc.intelligence.ollama_gateway import OllamaLLMGateway
from itc.intelligence.schemas import VerdictExplanation

print("Imports OK")

Imports OK


## 2. Load the rule catalogue (Layer 2)

Six YAML-defined rules across Section 16(2), 16(4), 17(5), and Rule 36(4) — the only place an eligibility decision is made.

In [2]:
catalogue = load_catalogue("itc/rules/catalogue")
print(f"Loaded {len(catalogue.rules)} rules: {list(catalogue.rules.keys())}")

Loaded 7 rules: ['rule_36_4_date_gate', 'section_16_2_a', 'section_16_2_b', 'section_16_2_c', 'section_16_2_d', 'section_16_4_time_bar', 'section_17_5_blocked']


## 3. Load synthetic fixtures

Synthetic GSTR-2B (government portal data) and purchase register (tenant's own ledger), generated with `faker`, including deliberately injected mismatches (~5% invoices unfiled by the vendor, 5-10% amount drift) to exercise every rule branch.

In [3]:
gstr2b_entries = parse_gstr2b(open("fixtures/gstr2b_tenant_a_062026.json", "rb").read())

mapping = mapping_from_tenant_profile("tenant_a", {
    "vendor_name": "Party Name",
    "vendor_gstin": "Party GSTIN",
    "invoice_number": "Voucher No",
    "invoice_date": "Voucher Date",
    "item_description": "Item Name",
    "taxable_amount": "Taxable Value",
    "gst_amount": "GST Amount",
    "tax_period": "Period",
})
register = parse_register(open("fixtures/purchase_register_tenant_a.xlsx", "rb").read(), mapping)

print(f"GSTR-2B entries: {len(gstr2b_entries)}")
print(f"Purchase register rows: {len(register.rows)}")

GSTR-2B entries: 500
Purchase register rows: 459


## 4. Reconcile — match invoices, evaluate against the rule engine

Every row is matched against GSTR-2B (exact key, then fuzzy invoice-number match), converted into `InvoiceFacts`, and passed through `evaluate()` — the single point in the whole system where an eligibility `Verdict` is produced. Only non-`eligible` verdicts become `Case`s below; clean matches pass through silently.

In [4]:
cases = reconcile("tenant_a", gstr2b_entries, register.rows, catalogue, as_of_date=date.today())

print(f"{len(cases)} flagged out of {len(register.rows)} total invoices "
      f"({len(register.rows) - len(cases)} eligible, not shown)")

25 flagged out of 459 total invoices (434 eligible, not shown)


## 5. LLM explanation (Layer 1)

For each flagged case, the local Mistral model explains the *already-decided* verdict in plain English. The LLM receives the verdict and reasoning chain as fixed input — it cannot change or re-derive the decision, only narrate it. `OllamaLLMGateway.call()` validates the model's JSON output against `VerdictExplanation` and retries once on a schema mismatch.

In [5]:
gateway = OllamaLLMGateway()  # one instance -> all traces accumulate in gateway.traces

for case in cases:
    print(f"\n{case.invoice_number} \u2192 {case.verdict.verdict.value}")
    explanation = gateway.call(
        task="explain_verdict",
        context={
            "invoice_number": case.invoice_number,
            "verdict": case.verdict.verdict.value,
            "reason_chain": [r.message for r in case.verdict.reason_chain],
        },
        tenant_id=case.tenant_id,
        response_schema=VerdictExplanation,
    )
    print(explanation.explanation)

print(f"\n\n--- Summary: {len(cases)} flagged out of {len(register.rows)} total invoices "
      f"({len(register.rows) - len(cases)} eligible, not shown) ---")


BILL-2019-3518 → ineligible
The invoice is deemed ineligible for Input Tax Credit (ITC) as per Section 16(2)(aa) of the CGST Act 2017 because it was not present in GSTR-2B for the period 06/2026 and provisional ITC was abolished post-effectively on 01.01.2022 (Notif 40/2021-CT).

SL-1976-03945 → ineligible
The invoice is considered ineligible for Input Tax Credit (ITC) because it was not present in GSTR-2B for the period 06/2026, and the provisional ITC under Rule 36(4) was abolished as of 01.01.2022. As this situation occurred during a post-effective tax period (06/2026), the verdict is 'ineligible' according to Section 16(2)(aa) CGST Act 2017.

GST-2000-0263770 → ineligible
The invoice with number GST-2000-0263770 is ineligible because it was not present in the GSTR-2B for period 06/2026, and since Rule 36(4) provisional ITC was abolished on 01.01.2022 (Notif 40/2021-CT), tax period 06/2026 being post-effective leads to the verdict of ineligible as per Section 16(2)(aa) CGST Act 201

## 6. Audit trail

Every LLM call is traced (audit-first pattern from `AbstractLLMGateway`) — proof the SLM's role was explanation only, never decision, for every single case above.

In [6]:
print(f"{len(gateway.traces)} LLM calls traced this run")
for trace in gateway.traces[:3]:
    print(trace)

25 LLM calls traced this run
trace_id=UUID('f30dc567-83ea-4f91-baa9-030a681fe3ab') tenant_id='tenant_a' task='explain_verdict' context={'invoice_number': 'BILL-2019-3518', 'verdict': 'ineligible', 'reason_chain': ['Section 17(5) CGST Act 2017: satisfied.', 'Section 16(4) CGST Act 2017: satisfied.', 'Section 16(2)(a) CGST Act 2017: satisfied.', 'Section 16(2)(b) CGST Act 2017: satisfied.', 'Section 16(2)(c) CGST Act 2017: satisfied.', 'Invoice not present in GSTR-2B for period 06/2026; ITC provisional pending supplier reconciliation (Rule 36(4)).', 'Rule 36(4) provisional ITC abolished 01.01.2022 (Notif 40/2021-CT). Tax period 06/2026 is post-effective; verdict -> ineligible (Sec 16(2)(aa)).']} response_schema_name='VerdictExplanation' created_at=datetime.datetime(2026, 7, 10, 15, 14, 35, 186918, tzinfo=datetime.timezone.utc)
trace_id=UUID('8b08261e-9685-4280-94d3-76ad9778345a') tenant_id='tenant_a' task='explain_verdict' context={'invoice_number': 'SL-1976-03945', 'verdict': 'ineligibl

## 7.Various other verdicts from engine


In [7]:
scenarios = {
    "TEST-ELIGIBLE-01": ("eligible", InvoiceFacts(
        tenant_id="tenant_a", item_description_clean="office stationery",
        vendor_gstin="27AAAAA0000A1Z5", taxable_amount_inr=10000.0, gst_amount_inr=1800.0,
        tax_period="06/2026", present_in_gstr2b=True, supplier_filed_gstr1=True,
        as_of_date=date.today(),
    )),
    "TEST-INELIGIBLE-01": ("ineligible", InvoiceFacts(
        tenant_id="tenant_a", item_description_clean="office stationery",
        vendor_gstin="27AAAAA0000A1Z5", taxable_amount_inr=10000.0, gst_amount_inr=1800.0,
        tax_period="06/2026", present_in_gstr2b=False, supplier_filed_gstr1=True,
        as_of_date=date.today(),
    )),
    "TEST-TIMEBARRED-01": ("time_barred", InvoiceFacts(
        tenant_id="tenant_a", item_description_clean="office stationery",
        vendor_gstin="27AAAAA0000A1Z5", taxable_amount_inr=10000.0, gst_amount_inr=1800.0,
        tax_period="03/2025", present_in_gstr2b=True, supplier_filed_gstr1=True,
        as_of_date=date.today(),
    )),
    "TEST-BLOCKED-01": ("blocked", InvoiceFacts(
        tenant_id="tenant_a", item_description_clean="motor vehicle repair charges",
        vendor_gstin="27AAAAA0000A1Z5", taxable_amount_inr=10000.0, gst_amount_inr=1800.0,
        tax_period="06/2026", present_in_gstr2b=True, supplier_filed_gstr1=True,
        as_of_date=date.today(),
    )),
}

for synthetic_id, (expected, facts) in scenarios.items():
    verdict = evaluate(facts, catalogue)
    status = "✓" if verdict.verdict.value == expected else "✗ MISMATCH"
    print(f"\n{synthetic_id} → {verdict.verdict.value} (expected: {expected}) {status}")
    explanation = gateway.call(
        task="explain_verdict",
        context={
            "verdict": verdict.verdict.value,
            "reason_chain": [r.message for r in verdict.reason_chain],
        },
        tenant_id="tenant_a",
        response_schema=VerdictExplanation,
    )
    print(explanation.explanation)


TEST-ELIGIBLE-01 → eligible (expected: eligible) ✓
All eligibility conditions mentioned in Sections 17(5), 16(4), 16(2)(a), 16(2)(b), 16(2)(c), and 16(2)(d) of the CGST Act 2017 have been satisfied, resulting in an 'eligible' verdict.

TEST-INELIGIBLE-01 → ineligible (expected: ineligible) ✓
The verdict is 'ineligible'. This decision is based on Section 16(2)(aa) CGST Act 2017 due to the absence of an invoice in GSTR-2B for period 06/2026, which was post-effective. Previously, Rule 36(4) provisional ITC was applicable for this situation, but it was abolished on 01.01.2022 (Notif 40/2021-CT).

TEST-TIMEBARRED-01 → time_barred (expected: time_barred) ✓
The verdict is 'time_barred'. This means the Invoice for tax period 03/2025 is beyond its claim deadline (30 November following the financial year) as per Section 16(4) CGST Act 2017. The deadline passed on 2026-07-10, making the Input Tax Credit (ITC) claim invalid.

TEST-BLOCKED-01 → blocked (expected: blocked) ✓
The transaction for mot